# 실험 03: 스트리밍 출력 (Streaming)과 UX

## 1. 실험 제목과 목표
- **제목**: Blocking vs Streaming 방식 비교 및 Chunk 관찰 실험
- **목표**: `stream=True` 옵션을 통해 체감 응답 속도(TTFT)가 얼마나 극적으로 향상되는지 비교하고, 내부적으로 데이터 조각(Chunk)이 어떻게 전달되는지 파악합니다. 또한 스트리밍 시 발생할 수 있는 파싱 에러를 체험합니다.

## 2. 실험 개요
1. **비교 실험 1**: 일반 호출(Blocking)과 스트리밍(Streaming)의 초기 응답 시간 비교
2. **비교 실험 2**: 반환되는 데이터 Chunk의 구조 관찰
3. **비교 실험 3**: 출력 형식(마크다운 표 vs 일반 텍스트)에 따른 스트리밍 체감 비교
4. **실패 케이스**: 스트리밍 도중 JSON 파싱을 시도할 때 발생하는 에러 확인

## 3. 라이브러리 import

In [ ]:
import os
import time
import json
from dotenv import load_dotenv
from openai import OpenAI

## 4. 환경 변수 로드 및 client 생성

In [ ]:
load_dotenv()
client = OpenAI()

## 5. 비교 실험 1: Blocking vs Streaming 체감 속도 비교
똑같은 긴 글을 요청할 때, 사용자가 첫 글자를 보기까지 걸리는 시간(TTFT: Time To First Token)을 비교합니다.

In [ ]:
prompt = "한국의 사계절 특징에 대해 아주 자세하게 설명해줘."

print("=== [일반 호출 (Blocking)] ===")
start_block = time.time()
res_block = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)
end_block = time.time()
print(f"-> ⏳ 응답 완료까지 {end_block - start_block:.2f}초 동안 화면이 멈춰있습니다!")

print("\n=== [스트리밍 호출 (Streaming)] ===")
start_stream = time.time()
stream_res = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    stream=True
)

first_token_time = None
for chunk in stream_res:
    if first_token_time is None:
        first_token_time = time.time()
        print(f"\n-> ⚡ 첫 글자 응답까지 {first_token_time - start_stream:.2f}초 소요! (이후 타자 치듯 출력됨)")
        print("내용: ", end="")
    
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

end_stream = time.time()
print(f"\n-> 🏁 최종 완료까지 {end_stream - start_stream:.2f}초 소요")

## 6. 비교 실험 2: Chunk 데이터 뜯어보기
실제로 데이터가 어떤 묶음 단위로 들어오는지 관찰합니다.

In [ ]:
print("[Chunk 관찰하기]")
stream_chunk = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "1부터 10까지 세어봐."}],
    stream=True
)

for idx, chunk in enumerate(stream_chunk):
    content = chunk.choices[0].delta.content
    # content가 None인 경우는 보통 맨 마지막(finish_reason='stop')일 때입니다.
    if content is not None:
        # repr()을 사용해 공백과 줄바꿈 문자까지 시각적으로 확인합니다.
        print(f"Chunk {idx:02d}: {repr(content)}")

## 7. 비교 실험 3: 마크다운 표 스트리밍
단순 텍스트가 아닌 복잡한 구조(표)를 스트리밍할 때 어떻게 출력되는지 확인합니다.

In [ ]:
print("[표 스트리밍 테스트]")
stream_table = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "강아지와 고양이의 특징을 Markdown 표 형식으로 비교해줘."}],
    stream=True
)

for chunk in stream_table:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)
print("\n(표 구조도 | 기호를 포함해 글자 단위로 조각나서 전달되는 것을 볼 수 있습니다.)")

## 8. 실패/주의 케이스: JSON 파싱 에러
스트리밍은 문자열의 조각(Chunk)을 보내줍니다. 즉, 응답이 끝날 때까지는 '불완전한 JSON 문자열' 상태이므로 실시간 파싱이 불가능합니다.

In [ ]:
print("[JSON 실시간 파싱 시도]")
stream_json = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "JSON 형식으로만 답해. 예: {'result': 100}"},
        {"role": "user", "content": "50+50은?"}
    ],
    stream=True
)

partial_string = ""
try:
    for chunk in stream_json:
        content = chunk.choices[0].delta.content
        if content:
            partial_string += content
            print(f"현재 누적 문자열: {partial_string}")
            # 여기서 억지로 JSON 파싱을 시도합니다.
            parsed_data = json.loads(partial_string)
            print("파싱 성공:", parsed_data)
except Exception as e:
    print("\n🔥 에러 발생:", type(e).__name__, "-", e)
    print("-> 결과: '{\n  "res' 와 같은 불완전한 텍스트를 파싱하려 했기 때문에 JSONDecodeError가 발생합니다.")
    print("   스트리밍 응답을 JSON으로 변환하려면 전부 누적한 뒤 마지막에 파싱하거나 별도의 특수 라이브러리(iJSON 등)가 필요합니다.")

## 9. 결과 해석

1. **TTFT (Time To First Token)**: 사용자가 기다림에 지루함을 느끼지 않도록, 첫 글자가 화면에 찍히는 시간(TTFT)을 최소화하는 것이 서비스 UX의 핵심입니다.
2. **Chunk 데이터**: 단어 단위가 아니라 서브워드(Subword)나 철자 단위로 쪼개져서 들어오며, 프론트엔드와 연결할 때 이 Chunk들을 조립(concat)해서 화면에 랜더링해야 합니다.
3. **구조화 데이터와의 상충**: JSON 데이터를 반환받아 백엔드에서 DB에 저장하거나 로직을 처리해야 할 때는, 스트리밍보다 일반 호출(Blocking)이 안정적이고 파싱하기 좋습니다.

## 10. Notion 실험 로그

### 발견한 문제점
* 스트리밍 모드(`stream=True`)를 사용하면 챗봇처럼 타자 치는 효과를 주어 UX가 극적으로 상승하지만, 백엔드 로직 처리(JSON 데이터 변환 등) 시 완전한 문자열이 아니기 때문에 중간 처리가 매우 까다로움.
* Markdown 표 같은 구조물도 한 글자씩 오기 때문에 UI 컴포넌트(React 등)에서 실시간 랜더링 시 화면이 깜빡이거나 깨질 수 있음.

### 다음 개선 방향
* 용도 분리: 사용자에게 텍스트를 바로 보여주는 기능은 Streaming을 쓰고, 데이터를 추출(Structured Output)하여 시스템 내부적으로 쓰는 곳은 Blocking(일반 호출) 방식을 혼용하는 아키텍처를 설계해야 함.